# crypto

In [1]:
///- --test static

open testing
open file_system_operators

In [ ]:
open rust
open rust_operators
open sm'_operators

## fsharp

### sha256

In [ ]:
nominal sha256 = $'System.Security.Cryptography.SHA256'

inl sha256 () : sha256 =
    $'`sha256.Create' ()

### sha256_compute_hash

In [ ]:
inl sha256_compute_hash (x : sha256) (data : a i32 u8) : a i32 u8 =
    data |> $'!x.ComputeHash'

## rust

### get_file_hash'

In [ ]:
inl get_file_hash' (path : string) : result string string =
    inl path = path |> file_system.normalize_path
    inl exit_code, result =
        runtime.execution_options fun x => { x with
            command = $'$"pwsh -c \\\"(Get-FileHash \'{!path}\' -Algorithm SHA256).Hash\\\""'
        }
        |> runtime.execute_with_options
    if exit_code = 0
    then result |> sm'.to_lower |> Ok
    else result |> Error

In [2]:
///- --test
///> rust -d chrono encoding_rs encoding_rs_io regex sha2

inl file_name = "test.txt"
inl text = "\n"

inl temp_dir, disposable =
    #?*(file_name, text)
    |> crypto.hash_text
    |> file_system.create_temp_dir'
disposable |> use |> ignore
inl path = temp_dir </> file_name
text |> file_system.write_all_text path
path
|> get_file_hash'
|> resultm.get
|> _assert_eq "01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b"

17:19:01 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_475fc45d4d43e36176987485d59691bf9ef003902eefa1c29ef9dd4a6420dc2c/11673ef1-3afd-b48c-b456-054dcc9be5a8 }
17:19:01 2d runtime.execute_with_options / { file_name = pwsh; arguments = ["-c", "(Get-FileHash '/tmp/!create_temp_path_/spiral_475fc45d4d43e36176987485d59691bf9ef003902eefa1c29ef9dd4a6420dc2c/11673ef1-3afd-b48c-b456-054dcc9be5a8/test.txt' -Algorithm SHA256).Hash"] }
17:19:02 3v > 01BA4719C80B6FE911B091A7C05124B64EEECE964E09C058EF8F9805DACA546B
17:19:02 4v runtime.execute_with_options / result / { file_name = pwsh; exit_code = 0; std_trace_len = 64 }
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }

### sha256'

In [ ]:
nominal sha256' =
    `(
        rust_type
            `($'sha2_Sha256')
            `($'sha2::Sha256')
            "type sha2_Sha256 = class end"
    )

### new_sha256

In [ ]:
inl new_sha256 () : sha256' =
    (!\($'"true; let result : sha2::Sha256 = sha2::Digest::new()"') : bool) |> ignore
    !\($'"result"')

### hasher_update

In [ ]:
inl hasher_update forall el dim. (slice : rust.ref (am'.slice' el dim)) (hasher : sha256') : () =
    (!\($'"true; sha2::Digest::update(&mut !hasher, !slice)"') : bool) |> ignore

### hasher_finalize

In [ ]:
inl hasher_finalize (hasher : sha256') : rust.ref (am'.slice u8) =
    !\($'"&sha2::Digest::finalize(!hasher)"')

### hash_read

In [ ]:
inl hash_read data : resultm.result' string stream.io_error =
    inl reader = data |> stream.new_buf_reader
    (!\($'"true; let mut !reader = !reader"') : bool) |> ignore
    inl hasher = new_sha256 ()
    (!\($'"true; let mut !hasher = !hasher"') : bool) |> ignore
    real
        inl size = 1024
        inl zero = convert.convert `i32 `(convert.unativeint) 0
        inl buffer = am'.new_slice `u8 `@size 0u8
        rust.loop 0 fun () =>
            inl count = stream.buf_reader_read `u8 `@size buffer reader
            inl count = resultm.unwrap' `(convert.unativeint) `(stream.io_error) count
            inl _ =
                if (=.) `(convert.unativeint) count zero then
                    rust.break `(()) ()
                    true
                else false
            hasher_update `u8 `@size
                (
                    am'.slice_range `u8 `@size
                        (am'.Start `(convert.unativeint) zero)
                        (am'.End `(convert.unativeint)
                            ((fun _ => count) : (() -> convert.unativeint) -> convert.unativeint))
                        buffer
                )
                hasher
    hasher
    |> hasher_finalize
    |> am'.slice_to_vec
    |> am'.vec_map (sm'.format_hex' >> sm'.from_std_string)
    |> am'.from_vec_base
    |> am'.to_list_base'
    |> listm'.unbox
    |> sm'.concat_list (join "")
    |> Ok
    |> resultm.box

### get_file_hash

In [ ]:
inl get_file_hash (path : string) =
    inl path = path |> file_system.normalize_path
    inl file = path |> file_system.file_open |> resultm.unwrap'
    inl reader = file |> stream.new_buf_reader
    reader
    |> hash_read

In [3]:
///- --test
///> rust -d chrono regex sha2

inl file_name = join "test.txt"
inl text = "\n"

inl temp_dir, disposable =
    #?*(file_name, text)
    |> crypto.hash_text
    |> file_system.create_temp_dir'
disposable |> use |> ignore

inl path = temp_dir </> file_name
text |> file_system.write_all_text path

path
|> get_file_hash
|> resultm.unwrap'
|> _assert_eq "01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b"

17:19:03 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_0800a1de3caf3e576216365c97e31f81f1532bc4443f7e2a446a85b7202012be/11673ef1-3afd-b48c-b456-054dcc9be5a8 }
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }

In [4]:
///- --test
///> rust -d chrono regex sha2

inl file_name = join "test.txt"
inl text = ""

inl temp_dir, disposable =
    #?*(file_name, text)
    |> crypto.hash_text
    |> file_system.create_temp_dir'
disposable |> use |> ignore

inl path = temp_dir </> file_name
text |> file_system.write_all_text path
path
|> get_file_hash
|> resultm.unwrap'
|> _assert_eq "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"

17:18:56 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_7339ff3db8fdbcd32e92af7f3e8892ae609bb2dfcecc7b81f6fb3779d27ea4e2/536878b7-182d-4692-f1ac-23e55345bf6d }
__assert_eq { expected = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 }

## typescript

### create_hash

In [ ]:
inl create_hash (x : string) : any =
    global "import * as spiral_node_crypto from 'node:crypto';"
    $'spiral_node_crypto.createHash(!x)' : any

### hash_update

In [ ]:
inl hash_update (s : string) (x : any) : any =
    $'!x.update(!s, "utf8")' : any

### hash_digest

In [ ]:
inl hash_digest (s : string) (x : any) : string =
    $'!x.digest(!s)' : string

## python

### py_sha256

In [ ]:
nominal py_sha256 = any

### hashlib_sha256

In [ ]:
inl hashlib_sha256 () : py_sha256 =
    backend_switch {
        Python = fun () =>
            global "import hashlib"
            $'hashlib.sha256()' : py_sha256
    }

### sha256_update

In [ ]:
inl sha256_update (x : string) (sha256 : py_sha256) : py_sha256 =
    backend_switch {
        Python = fun () =>
            $'!sha256.update(!x)' : ()
    }
    sha256

### sha256_hexdigest

In [ ]:
inl sha256_hexdigest (sha256 : py_sha256) : string =
    backend_switch {
        Python = fun () =>
            $'!sha256.hexdigest()' : string
    }

## crypto

### hash_text

In [ ]:
let hash_text (~input : string) =
    run_target function
        | Fsharp => fun () =>
            inl sha256 = sha256 () |> use
            input
            |> sm'.utf8_get_bytes
            |> sha256_compute_hash sha256
            |> am.map (sm'.byte_to_string "x2")
            |> seq.of_array'
            |> sm'.concat (join "")
        | TypeScript => fun () =>
            create_hash "sha256"
            |> hash_update input
            |> hash_digest "hex"
        | Rust => fun () =>
            input
            |> sm'.utf8_get_bytes
            |> fun (a x) => x
            |> am'.to_vec
            |> stream.new_cursor
            |> hash_read
            |> resultm.unwrap'
        | Cuda => fun () =>
            hashlib_sha256 ()
            |> sha256_update (input |> sm'.encode_utf8)
            |> sha256_hexdigest
        | _ => fun () => null ()

In [5]:
///- --test
///> fsharp
///> cuda
///> rust -d sha2
///> typescript

"\n"
|> hash_text
|> _assert_eq "01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b"

""
|> hash_text
|> _assert_eq "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"

.fsx (fsharp)
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }
__assert_eq { expected = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 }
.py (cuda)
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }
__assert_eq { expected = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 }
0
.rs (rust)
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }
__assert_eq { expected = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 }
.ts (typescript)
__assert_eq { expected = 01ba4719c80b6fe911b091a7c05124b64eeece964e09c058ef8f9805daca546b }
__assert_eq { expected = e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855 }

### hash_to_port

In [ ]:
inl hash_to_port (text : string) : u16 =
    inl first_letter_code = text |> sm'.index 0i32 |> sm'.convert_to_utf32
    inl hash_part = text |> sm'.slice 0i32 7
    inl combined_value = convert.convert_i32_base 16 hash_part + first_letter_code |> convert.u16
    trace Verbose
        fun () => "crypto.hash_to_port"
        fun () => { first_letter_code hash_part combined_value }
    combined_value % 48128 + 1024

In [6]:
///- --test
///> fsharp
///> cuda
///> rust -d sha2
///> typescript

"\n"
|> hash_text
|> hash_to_port
|> _assert_eq 19273

.fsx (fsharp)
17:19:05 1v crypto.hash_to_port / { first_letter_code = 48; hash_part = 01ba4719; combined_value = 18249 }
__assert_eq { expected = 19273 }
.py (cuda)
17:19:07 1v crypto.hash_to_port / { first_letter_code = 48; hash_part = 01ba4719; combined_value = 18249 }
__assert_eq { expected = 19273 }
0
.rs (rust)
17:19:11 1v crypto.hash_to_port / { first_letter_code = 48; hash_part = 01ba4719; combined_value = 18249 }
__assert_eq { expected = 19273 }
.ts (typescript)
17:19:12 1v crypto.hash_to_port / { first_letter_code = 48; hash_part = 01ba4719; combined_value = 18249 }
__assert_eq { expected = 19273 }

## main

In [7]:
///> fsharp

inl main () =
    $'let hash_text x = !hash_text x' : ()
    $'let hash_to_port x = !hash_to_port x' : ()